Подключаем Google Disk, создаем папку, куда пойдет итоговая csv таблица:

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

drive_results_path = "/content/drive/MyDrive/nemo_results"
os.makedirs(drive_results_path, exist_ok=True)

Скачиваем необходимые зависимости:



In [ ]:
!pip install nemo_toolkit[asr] datasets soundfile pyannote.metrics pandas torchaudio

Авторизация на Hugging Face Hub:

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
  hf_token = userdata.get('HF_TOKEN')
  os.environ['HF_TOKEN'] = hf_token
  print("Token has downloaded from Colab's secrets !")

  print("Logging in HuggingFace...")
  login(token=hf_token)
except Exception as e:
  print(f"Error: {e}")
  hf_token = None

Создание переменных с путями, моделями и датасетом:

In [ ]:
import wget
import os
from omegaconf import OmegaConf

diarization_data_set = "diarizers-community/ami"
dataset_config = "ihm" # default: None
output_dir = "nemo_outputs"

os.makedirs("dataset_samples", exist_ok=True)
os.makedirs("results", exist_ok=True)

config_url = "https://raw.githubusercontent.com/NVIDIA/NeMo/v1.22.0/examples/speaker_tasks/diarization/conf/inference/diar_infer_meeting.yaml"
config = OmegaConf.load(wget.download(config_url, out='diar_infer_meeting.yaml'))

pretrained_speaker_model = 'titanet_large'
pretrained_vad = 'vad_multilingual_marblenet'
config.diarizer.out_dir = output_dir

Устанавливаем стандартные параметры для NeMo с NeuralDiarizer:

In [ ]:
config.diarizer.speaker_embeddings.model_path = pretrained_speaker_model
config.diarizer.speaker_embeddings.parameters.window_length_in_sec = [1.5, 1.25, 1.0, 0.75, 0.5]
config.diarizer.speaker_embeddings.parameters.shift_length_in_sec = [0.75, 0.625, 0.5, 0.375, 0.25]
config.diarizer.speaker_embeddings.parameters.multiscale_weights= [1, 1, 1, 1, 1]

config.diarizer.oracle_vad = False
config.diarizer.vad.model_path = pretrained_vad

config.diarizer.clustering.parameters.oracle_num_speakers = False
config.diarizer.clustering.parameters.max_num_speakers = 5

config.diarizer.msdd_model.model_path = 'diar_msdd_telephonic'

Определяем границы для метрики DER:

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, DiarizationPurity, DiarizationCoverage
from pyannote.metrics.detection import DetectionErrorRate

configs = {
    'strict': {'collar': 0.0, 'skip_overlap': False},
    'collar': {'collar': 0.25, 'skip_overlap': False},
    'no_ovl': {'collar': 0.0, 'skip_overlap': True},
    'clean':  {'collar': 0.25, 'skip_overlap': True}
}

metrics_vault = {}
for c_name, params in configs.items():
    metrics_vault[c_name] = {
        'der': DiarizationErrorRate(**params),
        'purity': DiarizationPurity(**params),
        'coverage': DiarizationCoverage(**params),
        'det': DetectionErrorRate(collar=params['collar']) # VAD is independent of overlap
    }

Закачиваем все аудио из выбранного датасета:

In [ ]:
dataset = load_dataset(f"{diarization_data_set}", name=dataset_config, split="test", streaming=True)

test_samples = dataset
a = 0
for idx, sample in enumerate(test_samples):
    audio = sample["audio"]
    audio_path = f"dataset_samples/sample_{idx}.wav"
    sf.write(audio_path, audio["array"], audio["sampling_rate"])
    a += len(audio["array"]) / audio["sampling_rate"]
print(a)

Для NeMo diarization нужно настроить manifest файл с метаинформацией, которую он использует при работе.
После настройки запускается такое же тестирование, как и с Pyannote. Метрики так же считаются через библиотеку pyannote.metrics, что очень удобно.

In [ ]:
from datasets import load_dataset
from datetime import datetime
import os
import soundfile as sf
from pyannote.core import Annotation, Segment
import json
from omegaconf import OmegaConf
import nemo.collections.asr as nemo_asr
import pandas as pd
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset = load_dataset(f"{diarization_data_set}", name=dataset_config, split="test", streaming=True)

test_samples = dataset
results = []

for idx, sample in enumerate(test_samples):
    audio = sample["audio"]
    audio_path = f"dataset_samples/sample_{idx}.wav"
    sf.write(audio_path, audio["array"], audio["sampling_rate"])

    reference = Annotation()
    rttm_ref_path = os.path.abspath(f"dataset_samples/ref_{idx}.rttm")
    with open(rttm_ref_path, "w") as f:
        for start, end, speaker in zip(sample['timestamps_start'], sample['timestamps_end'], sample['speakers']):
            duration = end - start
            line = f"SPEAKER sample_{idx} 1 {start:.3f} {duration:.3f} <NA> <NA> {speaker} <NA> <NA>\n"
            f.write(line)

            reference[Segment(start, end)] = speaker

    manifest_path = f"dataset_samples/manifest_{idx}.json"
    with open(manifest_path, "w") as f:
        entry = {
            "audio_filepath": audio_path,
            "offset": 0.0,
            "duration": len(audio["array"]) / audio["sampling_rate"],
            "label": "infer",
            "text": "-",
            "rttm_filepath": rttm_ref_path,
            "num_speakers": None,
        }
        f.write(json.dumps(entry) + "\n")

    config.diarizer.manifest_filepath = manifest_path

    cluster_model = nemo_asr.models.NeuralDiarizer(cfg=config).to(device)
    cluster_model.diarize()

    rttm_file = os.path.join(output_dir, 'pred_rttms', f"sample_{idx}.rttm")

    hypothesis = Annotation()
    if os.path.exists(rttm_file):
        with open(rttm_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 8:
                    start, duration, speaker = float(parts[3]), float(parts[4]), parts[7]
                    hypothesis[Segment(start, start + duration)] = speaker

    res = {'file': f"sample_{idx}"}
    print(f"================================================== New Test. audio: sample_{idx}.wav ==================================================")
    for c_name, m_group in metrics_vault.items():
        res[f'DER_{c_name}'] = m_group['der'](reference, hypothesis)
        res[f'Purity_{c_name}'] = m_group['purity'](reference, hypothesis)
        res[f'Coverage_{c_name}'] = m_group['coverage'](reference, hypothesis)
        res[f'DetER_{c_name}'] = m_group['det'](reference, hypothesis)

        components = m_group['der'].compute_components(reference, hypothesis)

        res[f'FA_{c_name}'] = components['false alarm']
        res[f'Miss_{c_name}'] = components['missed detection']
        res[f'Conf_{c_name}'] = components['confusion']
        res[f'Total_Speech_{c_name}'] = components['total']

        print("-" * 60)
        print("DER: ", res[f'DER_{c_name}'])
        print("Purity: ", res[f'Purity_{c_name}'])
        print("Coverage: ", res[f'Coverage_{c_name}'])
        print("DetER: ", res[f'DetER_{c_name}'])
        print(f"- {c_name} -")
        print("FA: ", res[f'FA_{c_name}'])
        print("Miss:", res[f'Miss_{c_name}'])
        print("Conf: ", res[f'Conf_{c_name}'])

    results.append(res)

current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
df.to_csv(f"results/nemo_metrics_{current_time}.csv", index=False)
df.to_csv(f"{drive_results_path}/nemo_metrics_{current_time}.csv", index=False)
print(".csv file saved successfully!")

Визуализация ошибок:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axisartist.axislines import Subplot
from matplotlib.transforms import blended_transform_factory


def plot_single_histogram(data, categories, colors, ylabel,
                          ylim=None, y_offset_text=1,
                          text_precision='.2f', figsize=(6, 5), title=None):
    fig = plt.figure(figsize=figsize)
    ax = Subplot(fig, 111)
    fig.add_subplot(ax)
    ax.axis["left"].set_axisline_style("-|>", size=1.5)
    ax.axis["bottom"].set_visible(False)
    ax.axis["top"].set_visible(False)
    ax.axis["right"].set_visible(False)
    ax.set_ylabel(ylabel, fontsize=11)
    bars = ax.bar(categories, data, color=colors, edgecolor='black', alpha=0.8, width=0.6)
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        bottom = 0
        top = max(data) * 1.15 if max(data) > 0 else 10
        ax.set_ylim(bottom, top)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + y_offset_text,
                f'{height:{text_precision}}%', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    for bar, cat in zip(bars, categories):
        x_center = bar.get_x() + bar.get_width() / 2.
        ax.text(x_center, -0.04, cat, transform=transform,
                ha='center', va='top', fontsize=10, fontweight='normal')
    xlim = ax.get_xlim()
    x_range = xlim[1] - xlim[0]
    ax.set_xlim(xlim[0] - 0.1 * x_range, xlim[1] + 0.2 * x_range)
    plt.subplots_adjust(bottom=0.2)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')
    return fig, ax


def compute_weighted_metrics(df, suffix):
    total_speech = df[f'Total_Speech_{suffix}'].sum()
    fa_sum = df[f'FA_{suffix}'].sum()
    miss_sum = df[f'Miss_{suffix}'].sum()
    conf_sum = df[f'Conf_{suffix}'].sum()

    fa_pct = (fa_sum / total_speech) * 100
    miss_pct = (miss_sum / total_speech) * 100
    conf_pct = (conf_sum / total_speech) * 100
    der_pct = ((fa_sum + miss_sum + conf_sum) / total_speech) * 100
    purity_weighted = (df[f'Purity_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    coverage_weighted = (df[f'Coverage_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    return {
        'fa': fa_pct,
        'miss': miss_pct,
        'conf': conf_pct,
        'der': der_pct,
        'purity': purity_weighted,
        'coverage': coverage_weighted
    }


df_base = pd.read_csv("/content/drive/MyDrive/nemo_results/nemo_metrics_20260326_165811.csv")
configs = ["strict", "collar", "no_ovl", "clean"]
for cfg in configs:
    base = compute_weighted_metrics(df_base, cfg)
    plot_single_histogram(
        data=[base['miss'], base['fa'], base['conf']],
        categories=['Missed', 'FA', 'Confusion'],
        colors=['#FF6B6B', '#4D96FF', '#6BCB77'],
        ylabel='Доля ошибок (%)',
        ylim=(0, max(base['miss'], base['fa'], base['conf']) * 1.15),
        y_offset_text=0.05,
        figsize=(6, 5),
    )
    plt.savefig(f'base_der_components_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plot_single_histogram(
        data=[base['purity'], base['coverage']],
        categories=['Purity', 'Coverage'],
        colors=['#FFD93D', '#A084CA'],
        ylabel='Значение метрики (%)',
        ylim=(0, 105),
        y_offset_text=0.04,
        text_precision='.1f',
        figsize=(5, 5),
    )
    plt.savefig(f'base_purity_coverage_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()